In [ ]:
High precision : Not many real emails predicted as spam
High recall : Predicted most spam emails correctly

### KNN Performance

In [ ]:
A confusion matrix is a table that is often used to describe the performance of a classification model
(or "classifier") on a set of test data for which the true values are known. 

In [ ]:
* sklearn.model_selection.train_test_split and sklearn.neighbors.KNeighborsClassifier have already been imported

Import necessary modules
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

# Create training and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.4, random_state=42)

# Instantiate a k-NN classifier: knn
knn = KNeighborsClassifier(n_neighbors=6)

# Fit the classifier to the training data
knn.fit(X_train, y_train)

# Predict the labels of the test data: y_pred
y_pred = knn.predict(X_test)

# Confusion matrix and classification report (Precion,recall,f1-score,support)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


In [ ]:
Confusion matrix shows the values of performance;
[
[175,30]
[35,67]     175 = actual no,predicted no   67=actual yes,predicted yes
]

### Logistic Regression Performance

###### Despite its name, Logistic regression is used in classification models, not regression models !!

In [ ]:
# Import the necessary modules
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

# Create training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.4, random_state=42)

# Create the classifier: logreg
logreg = LogisticRegression()

# Fit the classifier to the training data
logreg.fit(X_train, y_train)

# Predict the labels of the test set: y_pred
y_pred = logreg.predict(X_test)

# Compute and print the confusion matrix and classification report
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

#### Plotting an ROC curve

In [ ]:
Classification reports and confusion matrices are great methods to quantitatively evaluate model performance,,
while ROC curves provide a way to visually evaluate models.

In [ ]:
Most classifiers in scikit-learn have a .predict_proba() method which returns the probability of a given sample 
being in a particular class. 

In [ ]:
# after the regression model and fitting;


# Import necessary modules
from sklearn.metrics import roc_curve

# Compute predicted probabilities: y_pred_prob
y_pred_prob = logreg.predict_proba(X_test)[:,1]

# Generate ROC curve values: fpr (false pozitive rate), tpr (true pozitive rate), thresholds
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)

# Plot ROC curve
plt.plot([0, 1], [0, 1], 'k--')
plt.plot(fpr, tpr)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.show()

In [ ]:
Precision =  TP / (TP + FP)

Recall = TP / (TP + FN)

### AUC computation

In [ ]:
 If the AUC is greater than 0.5, the model is better than random guessing. Always a good sign!

In [ ]:
# Import necessary modules
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score

# Compute predicted probabilities: y_pred_prob
y_pred_prob = logreg.predict_proba(X_test)[:,1]

# Compute and print AUC score
print("AUC: {}".format(roc_auc_score(y_test, y_pred_prob)))

# Compute cross-validated AUC scores: cv_auc
cv_auc = cross_val_score(logreg, X, y, cv=5, scoring='roc_auc')

# Print list of AUC scores
print("AUC scores computed using 5-fold cross-validation: {}".format(cv_auc))

### Hyperparameter tuning - GridSearchCV

In [ ]:
Like the alpha parameter of lasso and ridge regularization that you saw earlier, logistic regression also has a
regularization parameter: C. C controls the inverse of the regularization strength, and this is what you will tune
in this exercise. A large C can lead to an overfit model, while a small C can lead to an underfit model.

In [ ]:
# Import necessary modules
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# Setup the hyperparameter grid
c_space = np.logspace(-5, 8, 15)
param_grid = {'C': c_space}

# Instantiate a logistic regression classifier: logreg
logreg = LogisticRegression()

# Instantiate the GridSearchCV object: logreg_cv
logreg_cv = GridSearchCV(logreg, param_grid, cv=5)

# Fit it to the data
logreg_cv.fit(X, y)

# Print the tuned parameter and score
print("Tuned Logistic Regression Parameters: {}".format(logreg_cv.best_params_))
print("Best score is {}".format(logreg_cv.best_score_))


### Hyperparameter tuning - RandomizedSearchCV - Decision Trees

In [ ]:
GridSearchCV can be computationally expensive, especially if you are searching over a large hyperparameter space
and dealing with multiple hyperparameters. A solution to this is to use RandomizedSearchCV, in which not all
hyperparameter values are tried out. Instead, a fixed number of hyperparameter settings is sampled from specified 
probability distributions. 

In [ ]:
Decision trees have many parameters that can be tuned, such as max_features, max_depth, and min_samples_leaf: 
This makes it an ideal use case for RandomizedSearchCV.

In [ ]:
# Import necessary modules
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier


# Setup the parameters and distributions to sample from: param_dist
param_dist = {"max_depth": [3, None],
              "max_features": randint(1, 9),
              "min_samples_leaf": randint(1, 9),
              "criterion": ["gini", "entropy"]}

# Instantiate a Decision Tree classifier: tree
tree = DecisionTreeClassifier()

# Instantiate the RandomizedSearchCV object: tree_cv
tree_cv = RandomizedSearchCV(tree, param_dist, cv=5)

# Fit it to the data
tree_cv.fit(X, y)

# Print the tuned parameters and score
print("Tuned Decision Tree Parameters: {}".format(tree_cv.best_params_))
print("Best score is {}".format(tree_cv.best_score_))

### Hold-out Set for final evaluation :with Classification

In [ ]:
We make this to see how well my model perform on never before seen data

In [ ]:
n addition to C, logistic regression has a 'penalty' hyperparameter which specifies whether to use 'l1' or 'l2'
regularization. Your job in this exercise is to create a hold-out set, tune the 'C' and 'penalty' hyperparameters 
of a logistic regression classifier using GridSearchCV on the training set.

In [ ]:
# Import necessary modules
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# Create the hyperparameter grid
c_space = np.logspace(-5, 8, 15)
param_grid = {'C': c_space, 'penalty': ['l1', 'l2']}

# Instantiate the logistic regression classifier: logreg
logreg = LogisticRegression()

# Create train and test sets
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.4,random_state=42)

# Instantiate the GridSearchCV object: logreg_cv
logreg_cv = GridSearchCV(logreg,param_grid,cv=5)

# Fit it to the training data
logreg_cv.fit(X_train,y_train)

# Print the optimal parameters and best score
print("Tuned Logistic Regression Parameter: {}".format(logreg_cv.best_params_))
print("Tuned Logistic Regression Accuracy: {}".format(logreg_cv.best_score_))


### Hold-out Set for final evaluation :with Regression

In [ ]:
Lasso used the L1 penalty to regularize, while ridge used the L2 penalty. There is another type of regularized 
regression known as the elastic net. In elastic net regularization, the penalty term is a linear combination of the L1
and L2 penalties.

In scikit-learn, this term is represented by the 'l1_ratio' parameter: An 'l1_ratio' of 1 corresponds to an L1 penalty,
and anything lower is a combination of L1 and L2.

In [ ]:
# Import necessary modules
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split

# Create train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.4, random_state=42)

# Create the hyperparameter grid
l1_space = np.linspace(0, 1, 30)
param_grid = {'l1_ratio': l1_space}

# Instantiate the ElasticNet regressor: elastic_net
elastic_net = ElasticNet()

# Setup the GridSearchCV object: gm_cv
gm_cv = GridSearchCV(elastic_net, param_grid, cv=5)

# Fit it to the training data
gm_cv.fit(X_train, y_train)

# Predict on the test set and compute metrics
y_pred = gm_cv.predict(X_test)
r2 = gm_cv.score(X_test, y_test)
mse = mean_squared_error(y_test, y_pred)
print("Tuned ElasticNet l1 ratio: {}".format(gm_cv.best_params_))
print("Tuned ElasticNet R squared: {}".format(r2))
print("Tuned ElasticNet MSE: {}".format(mse))